<a href="https://colab.research.google.com/github/07Akshaya/Statistical-Learning-e22019/blob/main/assignment7b_e22019.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Q. Bayesian Estimation of a User Ability Parameter from Item Responses

#### Task 1: Visualizing the Mechanics

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Define the 2PL Item Response Function
def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Generate a range of latent ability values (theta)
theta_vals = np.linspace(-6, 6, 300)

# Define configurations to plot
curves = [
    {"a": 0.5, "b": 0, "line_style": "dash"},
    {"a": 1.5, "b": -2, "line_style": "solid"},
    {"a": 1.5, "b": 0, "line_style": "solid"},
    {"a": 1.5, "b": 2, "line_style": "solid"},
]

# Create the Plotly figure
fig = go.Figure()

for curve in curves:
    a = curve["a"]
    b = curve["b"]
    style = curve["line_style"]

    # Calculate probabilities
    p_vals = p_i(theta_vals, a, b)

    # Add trace to the plot
    fig.add_trace(go.Scatter(
        x=theta_vals,
        y=p_vals,
        mode='lines',
        name=f"a = {a}, b = {b}",
        line=dict(dash=style, width=2.5)
    ))

# Customize layout
fig.update_layout(
    title={
        'text': "Two-Parameter Logistic (2PL) Item Response Curves",
        'y': 0.9,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    xaxis_title="Latent Ability (theta)",
    yaxis_title="Probability of Correct Response P(Y_i=1|theta)",
    xaxis=dict(range=[-6, 6], gridcolor='rgba(0,0,0,0.1)'),
    yaxis=dict(range=[0, 1.05], gridcolor='rgba(0,0,0,0.1)'),
    template="plotly_white",
    legend=dict(
        yanchor="top",
        y=0.95,
        xanchor="left",
        x=0.05,
        bgcolor="rgba(255,255,255,0.8)"
    )
)

# Display the interactive plot
fig.show()


Moving $b_i$ shifts the logistic item response curve horizontally along the ability axis. Increasing the value of $b_i$ shifts the curve to the **right**, indicating that a higher level of latent ability $\theta$ is required for a user to maintain the same probability of answering the question correctly.



#### Task 2: Sequential Likelihood Contribution


$$L(y_k \mid \theta) = p_k(\theta)^{y_k} (1 - p_k(\theta))^{1 - y_k}$$

Assuming conditional independence of item responses given $\Theta = \theta$, $y^{(k)} = (y_1, y_2, \dots, y_k)$


$$L(y^{(k)} \mid \theta) = \prod_{i=1}^k [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$



#### Task 3: Mathematical Formulation of the Running Update




$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$$




$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto \left[ p_k(\theta)^{y_k} (1 - p_k(\theta))^{1 - y_k} \right] f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$$



#### Task 4: Dynamic Shifting

When a user answers an item correctly ($y_k = 1$), the likelihood contribution simplifies directly to $L(y_k \mid \theta) = p_k(\theta)$. For a highly difficult item characterized by a large $b_k$, the item response function $p_k(\theta)$ manifests as a steep, right-shifted logistic curve that remains near zero for lower values of $\theta$ and scales toward one at elevated levels of $\theta$.

Multiplying the prior state density $f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$ by this heavily asymmetric likelihood profile suppresses the probability density across the lower domain of $\theta$ while preserving or amplifying it at the higher end. This mathematical modulation forces the peak (Maximum A Posteriori estimate) of the updated posterior density distribution to shift sharply to the **right** relative to the previous step.



#### Task 5: Tracking Certainty and Sharpness

The discrimination parameter $a_k$ dictates the mathematical slope and informational payload of the current item's likelihood function around its difficulty value $b_k$.

*   **Very Large $a_k$:** The logistic curve becomes extremely steep, acting as a highly precise threshold indicator. An observed response yields an immense informational update that rapidly eliminates uncertainty, narrowing the posterior variance and producing a highly focused, sharp density curve.


*   **Very Small $a_k$:** The logistic curve remains flat and widespread across the ability axis, evaluating almost uniformly across the grid. Because the item possesses minimal capacity to differentiate between distinct ability thresholds, it yields a negligible informational update, leaving the variance and sharpness of the posterior distribution virtually unaltered.





#### Task 6: Numerical Implementation of a Running Grid

*   **Grid Definition:** Establish a fixed, bounded coordinate vector of $M$ finely spaced latent ability points across a relevant physical domain ( $\theta \in [-5, 5]$).


*   **Prior Initialization:** Evaluate the standard normal probability density function across the grid coordinates to serve as the initial prior distribution state: $\text{posterior} = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right)$.


*   **Likelihood Computation:** Upon observing response $y_k$, compute the probability array $p_k(\theta) = \frac{1}{1 + \exp(-a_k(\theta - b_k))}$ across the fixed grid. Calculate the single-step likelihood vector element-wise: $\text{likelihood} = (p_k(\theta))^{y_k} \cdot (1 - p_k(\theta))^{(1 - y_k)}$.


*   **Proportional Update:** Construct the unnormalized posterior array via element-wise multiplication of the preceding posterior state and the new likelihood vector: $\text{posterior}_{\text{unnorm}} = \text{posterior} \times \text{likelihood}$.


*   **Sequential Normalization Step:** Apply the trapezoidal rule numerical integration over the fixed grid array to isolate the local normalizing constant:

$$\text{integral} = \text{np.trapezoid(posterior}_{\text{unnorm}}\text{, theta\_grid)}$$




Divide the unnormalized array by this scalar constant to enforce that the total area beneath the operational density curve integrates perfectly to one:

$$\text{posterior} = \frac{\text{posterior}_{\text{unnorm}}}{\text{integral}}$$





In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

np.random.seed(42)
theta_true = 0.75
n_items = 20
theta_grid = np.linspace(-5, 5, 1000)

# Generate random items
a_params = np.random.uniform(0.5, 2.0, size=n_items)
b_params = np.random.normal(0, 1, size=n_items)

# Tracking
running_bayes = [0.0]
running_map = [0.0]
steps = list(range(n_items + 1))
current_posterior = stats.norm.pdf(theta_grid, 0, 1)

for k in range(n_items):
    # Simulate response
    p_success = 1 / (1 + np.exp(-a_params[k] * (theta_true - b_params[k])))
    y_k = 1 if np.random.uniform(0, 1) < p_success else 0

    # Update likelihood and normalize
    prob_grid = 1 / (1 + np.exp(-a_params[k] * (theta_grid - b_params[k])))
    likelihood = (prob_grid ** y_k) * ((1 - prob_grid) ** (1 - y_k))
    current_posterior *= likelihood
    current_posterior /= np.trapezoid(current_posterior, theta_grid)

    # Store estimators
    running_bayes.append(np.trapezoid(theta_grid * current_posterior, theta_grid))
    running_map.append(theta_grid[np.argmax(current_posterior)])

# Visualize
fig = go.Figure()
fig.add_hline(y=theta_true, line_dash="dash", line_color="red", name="True Ability")
fig.add_trace(go.Scatter(x=steps, y=running_bayes, mode='lines+markers', name='Posterior Mean'))
fig.add_trace(go.Scatter(x=steps, y=running_map, mode='lines+markers', name='MAP Estimate'))
fig.update_layout(title="Convergence of Latent Ability Estimators", xaxis_title="Item (k)", yaxis_title="θ", template="plotly_white")
fig.show()

As k increases, the distance between the estimators and θ
true
  generally decreases as they converge toward the true value
. This narrowing and convergence imply that the platform's confidence in its measurement increases as more evidence is accumulated over time

###Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

#### Task 1: Structural Probability and Properties

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Generate a dense grid over the domain [0, 1]
theta_domain = np.linspace(0, 1, 500)

# Define the parameter pairs for the Beta distribution
states = [
    {"alpha": 1, "beta": 1, "name": "Uninformative State (1, 1)", "color": "gray"},
    {"alpha": 2, "beta": 8, "name": "Right-Skewed State (2, 8)", "color": "blue"},
    {"alpha": 8, "beta": 2, "name": "Left-Skewed State (8, 2)", "color": "orange"}
]

fig1 = go.Figure()

for state in states:
    a, b = state["alpha"], state["beta"]
    # Compute Beta PDF values
    pdf_vals = stats.beta.pdf(theta_domain, a, b)

    fig1.add_trace(go.Scatter(
        x=theta_domain,
        y=pdf_vals,
        mode='lines',
        name=state["name"],
        line=dict(color=state["color"], width=2.5)
    ))

# Adjust layout options
fig1.update_layout(
    title={
        'text': "Probability Density Function (PDF) of Beta Distributions",
        'y': 0.9, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Click-Through Rate (theta)",
    yaxis_title="Density",
    template="plotly_white",
    xaxis=dict(range=[0, 1], gridcolor='rgba(0,0,0,0.05)'),
    yaxis=dict(gridcolor='rgba(0,0,0,0.05)'),
    legend=dict(yanchor="top", y=0.95, xanchor="right", x=0.95)
)

fig1.show()


The hyperparameters $\alpha$ and $\beta$ function as prior pseudo-counts of successes (clicks) and failures (non-clicks). Altering their balance shifts the distribution's center of mass:

* When $\alpha = \beta = 1$, the mass is uniformly distributed across the entire domain, representing total uncertainty.


* When $\beta > \alpha$ (e.g., 2, 8), the center of mass shifts toward 0, creating a **right-skewed** distribution because the implied preponderance of non-clicks concentrates probability density at lower click-through rates.


* When $\alpha > \beta$ (e.g., 8, 2), the center of mass shifts toward 1, creating a **left-skewed** distribution because the weight of implied clicks concentrates the density at higher conversion rates.





#### Task 2: Sequential Likelihood and Joint History

The mathematical likelihood contribution $L(y_k \mid \theta)$ of a single isolated interaction $y_k \in \{0, 1\}$ at step $k$ is given by the Bernoulli probability mass function:


$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

Assuming each user interaction is conditionally independent given the true parameter $\Theta = \theta$, the joint likelihood function for the running history vector $y^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of individual updates:


$$L(y^{(k)} \mid \theta) = \prod_{i=1}^k \theta^{y_i} (1 - \theta)^{1 - y_i} = \theta^{\sum_{i=1}^k y_i} (1 - \theta)^{k - \sum_{i=1}^k y_i}$$

If we let $C_k = \sum_{i=1}^k y_i$ represent the total cumulative clicks observed up to step $k$, the joint likelihood condenses to:


$$L(y^{(k)} \mid \theta) = \theta^{C_k} (1 - \theta)^{k - C_k}$$



#### Task 3: Closed-Form Analytical Updates (Conjugacy)

According to Bayes' Theorem,


$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$$



$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right] \cdot \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$



$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$




$$\alpha_k = \alpha_{k-1} + y_k$$

$$\beta_k = \beta_{k-1} + (1 - y_k)$$




$$\mathbb{E}[\Theta \mid Y^{(k)} = y^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k} = \frac{\alpha_{k-1} + y_k}{\alpha_{k-1} + \beta_{k-1} + 1}$$



#### Task 4: Dynamic Shifting Mechanics

*   **Observed Click ($y_k = 1$):** Additively increments $\alpha_k$ by 1 while leaving $\beta_k$ unchanged. This increases the power of the $\theta$ term in the polynomial kernel, driving the probability mass and the mode of the distribution toward the **right** (closer to 1.0).


*   **Observed Non-Click ($y_k = 0$):** Additively increments $\beta_k$ by 1 while leaving $\alpha_k$ unchanged. This increases the power of the $(1-\theta)$ term, shifting the distribution's peak toward the **left** (closer to 0.0).





In non-conjugate architectures—such as the 2PL IRT model—the product of the likelihood and the prior results in a structural mathematical form that does not simplify to a recognized distribution family. Because the normalization constant cannot be calculated analytically via closed-form arithmetic adjustments, the system is forced to evaluate the entire continuous parameter space over a dense, discretized numerical grid, utilizing numerical integration methods (e.g., the trapezoidal rule) at every single step.

Conversely, this Beta-Binomial conjugate framework bypasses numerical grid maintenance completely. The updating mechanism is compressed into fast, exact scalar additions, allowing the continuous PDF to be perfectly tracked using only two parameters.



#### Task 5: Running Point Estimators

The exact closed-form point estimators evaluated at step $k$ are defined as:

### 1. Running Posterior Mean ($\hat{\theta}_{\text{Bayes}}^{(k)}$)

$$\hat{\theta}_{\text{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$$

### 2. Running Maximum A Posteriori ($\hat{\theta}_{\text{MAP}}^{(k)}$)

Given $\alpha_k > 1$ and $\beta_k > 1$ (the mode of the distribution):


$$\hat{\theta}_{\text{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$$



#### Task 6: Performance Tracking and Convergence Analysis

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Set seed for reproducible simulation
np.random.seed(42)

# 1. Simulation Parameters
theta_true = 0.35
n_impressions = 100

# Initialize prior parameters to Uniform state (alpha_0 = 1, beta_0 = 1)
alpha_curr = 1
beta_curr = 1

# Performance tracking vectors
running_bayes_ctr = [alpha_curr / (alpha_curr + beta_curr)]
# For alpha=1, beta=1, the uniform distribution mode can be initialized at 0.5
running_map_ctr = [0.5]
steps = list(range(n_impressions + 1))

# 2. Sequential Simulation Loop
for k in range(1, n_impressions + 1):
    # Simulate a Bernoulli trial based on true conversion rate
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0

    # Perform closed-form conjugate algebraic updates
    alpha_curr += y_k
    beta_curr += (1 - y_k)

    # Evaluate point estimators
    bayes_est = alpha_curr / (alpha_curr + beta_curr)
    map_est = (alpha_curr - 1) / (alpha_curr + beta_curr - 2) if (alpha_curr + beta_curr - 2) > 0 else 0.5

    running_bayes_ctr.append(bayes_est)
    running_map_ctr.append(map_est)

# 3. Visualization via Plotly
fig2 = go.Figure()

# Static horizontal reference line for true CTR
fig2.add_hline(
    y=theta_true,
    line_dash="dash",
    line_color="red",
    line_width=2,
    annotation_text=f"True CTR ({theta_true})",
    annotation_position="bottom right"
)

# Plot running Bayes Mean
fig2.add_trace(go.Scatter(
    x=steps, y=running_bayes_ctr,
    mode='lines',
    name='Posterior Mean (theta_Bayes)',
    line=dict(color='blue', width=2)
))

# Plot running MAP
fig2.add_trace(go.Scatter(
    x=steps, y=running_map_ctr,
    mode='lines',
    name='MAP Estimate (theta_MAP)',
    line=dict(color='green', width=2, dash='dot')
))

# Adjust plot styles
fig2.update_layout(
    title={
        'text': "Conjugate Sequential Beta-Binomial Estimation Progression",
        'y': 0.9, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Number of Impressions (k)",
    yaxis_title="CTR Estimate",
    template="plotly_white",
    xaxis=dict(range=[0, n_impressions]),
    yaxis=dict(range=[0, 1]),
    hovermode="x unified"
)

fig2.show()



As the sampling size $k$ approaches 100, the distance between both point estimators and the true parameter value $\theta_{\text{true}} = 0.35$ systematically shrinks, displaying clean asymptotic convergence. Early fluctuations are regularized as the cumulative empirical evidence grows.

This behavior highlights a foundational tenet of Bayesian updating: the accumulation of objective data progressively dominates the initial prior selection. As $k \to 100$, the specific values of $\alpha_0$ and $\beta_0$ carry minimal mathematical weight relative to the large observed counts of clicks ($C_k$) and non-clicks ($k - C_k$). The choice of an uninformative prior serves as an unbiased starting point, but the framework successfully overcomes initial assumptions to isolate the true underlying data-generating state with increasing certainty.

### Bayesian Estimation of a Structural Probability for Mechanical Degradation

## Task 1: Prior Belief Boundaries

### Analytical Expected Value

The initial prior distribution over the remaining stiffness efficiency factor is modeled as:


$$\Theta^{(0)} \sim \text{Beta}(\alpha, \beta), \quad \alpha = 8, \quad \beta = 1.5$$

The expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ is calculated analytically as:


$$\mathbb{E}[\Theta^{(0)}] = \frac{\alpha}{\alpha + \beta} = \frac{8}{8 + 1.5} = \frac{8}{9.5} \approx 0.8421$$

### Engineering Suitability

*   **Physical Meaning of Boundaries:** $\theta = 1.0$ represents a completely pristine, undamaged component, while $\theta \to 0$ represents critical failure.
*   **Prior Distribution Shape:** The Mode of the $\text{Beta}(8, 1.5)$ distribution is:

$$\text{Mode}[\Theta^{(0)}] = \frac{\alpha - 1}{\alpha + \beta - 2} = \frac{7}{7.5} \approx 0.9333$$


*   **Justification:** This highly skewed distribution concentrates the vast majority of its probability mass near $1.0$. It represents the realistic engineering assumption that a newly deployed or newly inspected component is highly likely to be healthy (pristine), while still leaving a small, non-zero probability of pre-existing micro-defects or minor degradation.

## Task 2: Structural Likelihood Formulation

### Single Measurement Likelihood

Given the physics model $y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}$ with $\epsilon_k \sim \mathcal{N}(0, \sigma^2)$, the measurement $y_k$ conditional on $\theta$ is log-normally distributed. Applying the change of variables formula:


$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp \left( -\frac{(\ln(y_k) - \ln(\theta \cdot K_{\text{nominal}}))^2}{2\sigma^2} \right)$$

### Joint Likelihood Function

Assuming that measurement noise terms $\epsilon_1, \dots, \epsilon_k$ are independent and identically distributed over time, the joint likelihood of the historical vector $y^{(k)} = (y_1, \dots, y_k)$ is the product of individual likelihood contributions:


$$L(y^{(k)} \mid \theta) = \prod_{i=1}^k L(y_i \mid \theta) = \left( \frac{1}{\sigma \sqrt{2\pi}} \right)^k \left( \prod_{i=1}^k \frac{1}{y_i} \right) \exp \left( -\sum_{i=1}^k \frac{(\ln(y_i) - \ln(\theta \cdot K_{\text{nominal}}))^2}{2\sigma^2} \right)$$

## Task 3: Mathematical Formulation of the Non-Conjugate Grid Update

### Non-Conjugacy Analysis

A conjugate prior yields a posterior distribution belonging to the same parametric family as the prior.

*   The prior distribution has a polynomial-algebraic kernel: $\theta^{\alpha-1} (1-\theta)^{\beta-1}$.
*   The log-normal likelihood has a kernel containing $\ln(\theta)$ nested within a squared exponent: $\exp\left(-\frac{(\ln\theta - C)^2}{2\sigma^2}\right)$.

Multiplying these functional forms yields:


$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto \theta^{\alpha-1} (1-\theta)^{\beta-1} \exp\left( -\frac{(\ln(y_k) - \ln(\theta \cdot K_{\text{nominal}}))^2}{2\sigma^2} \right)$$

This combined mathematical structure cannot be simplified or mapped to any standard, named probability density function. Hence, the prior and likelihood are non-conjugate, making analytical updates impossible.

### Recursive Relationship

At any inspection milestone $k$, the posterior is computed recursively by treating the previous posterior $f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$ as the current prior:


$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$$

## Task 4: Running Point Estimates

Since the posterior is non-conjugate, point estimators must be defined using definite integrals over the physical boundary domain $(0, 1]$:

### Running Posterior Mean ($\hat{\theta}_{\text{Bayes}}^{(k)}$)

$$\hat{\theta}_{\text{Bayes}}^{(k)} = \mathbb{E}[\Theta \mid Y^{(k)} = y^{(k)}] = \frac{\int_{0}^{1} \theta \cdot L(y_k \mid \theta) \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)}) \, d\theta}{\int_{0}^{1} L(y_k \mid \theta) \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)}) \, d\theta}$$

### Running Maximum A Posteriori ($\hat{\theta}_{\text{MAP}}^{(k)}$)

$$\hat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta \in (0, 1]} \left[ L(y_k \mid \theta) \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)}) \right]$$

## Task 5: Algorithmic Grid Approximation and Normalization


1.  **Grid Discretization:** Choose a grid size $N$ (e.g., $N = 1000$) and generate $N$ equally-spaced points $\theta_j$ spanning $[0.01, 1.0]$. The lower bound is strictly set to $0.01$ rather than $0.0$ to avoid computational singularities associated with $\ln(0)$ in the likelihood calculation. Define the grid spacing:

$$\Delta\theta = \frac{1.0 - 0.01}{N - 1}$$


2.  **Prior Initialization:** Evaluate the prior density $f_{\Theta}^{(0)}(\theta)$ at all grid points $\theta_j$, storing them in a vector $\mathbf{f}^{(0)}$. Normalize this vector using the trapezoidal rule:

$$\mathbf{f}^{(0)} \leftarrow \frac{\mathbf{f}^{(0)}}{\sum_{j=1}^{N-1} \frac{f^{(0)}_j + f^{(0)}_{j+1}}{2} \Delta\theta}$$


3.  **Recursive Likelihood Update:** When a new measurement $y_k$ is acquired, compute the likelihood vector $\mathbf{L}_k$ at each grid point:

$$L_{k, j} = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp \left( -\frac{(\ln(y_k) - \ln(\theta_j \cdot K_{\text{nominal}}))^2}{2\sigma^2} \right)$$


4.  **Element-Wise Multiplication:** Compute the unnormalized posterior vector $\mathbf{\tilde{f}}^{(k)}$:

$$\tilde{f}^{(k)}_j = L_{k, j} \cdot f^{(k-1)}_j \quad \forall j \in \{1, 2, \dots, N\}$$


5.  **Sequential Trapezoidal Normalization:** Compute the normalizing integral $I$ and scale the posterior:

$$I = \sum_{j=1}^{N-1} \frac{\tilde{f}^{(k)}_j + \tilde{f}^{(k)}_{j+1}}{2} \Delta\theta, \quad f^{(k)}_j = \frac{\tilde{f}^{(k)}_j}{I} \quad \forall j \in \{1, 2, \dots, N\}$$

## Task 6: Performance Tracking and Degradation Convergence Analysis

### Python Simulation Script

In [5]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Set random seed for reproducibility
np.random.seed(24)

# =====================================================================
# CONFIGURATION & PARAMETERS
# =====================================================================
theta_true = 0.68       # True remaining stiffness efficiency of the beam (68%)
K_nominal = 50.0        # Nominal baseline stiffness of the pristine structure (kN/mm)
sigma = 0.15            # Sensor noise standard deviation (log-space)
n_sensor_readings = 15  # Timeline steps

# 1. Define a fine grid over the physical boundary [0.01, 1.0]
theta_grid = np.linspace(0.01, 1.0, 500)

# 2. Initialize Prior: Bounded Beta distribution reflecting an initially healthy beam
current_posterior = stats.beta.pdf(theta_grid, a=8, b=1.5)
# Normalize initial prior
current_posterior /= np.trapezoid(current_posterior, theta_grid)

# Initialize trackers for tracking estimators (Step 0)
bayes_estimates = [np.trapezoid(theta_grid * current_posterior, theta_grid)]
map_estimates = [theta_grid[np.argmax(current_posterior)]]

# Steps milestone tracking for plotting curves
milestones = [0, 1, 2, 5, 10, 15]

# Create Figure 1: Posterior Density Curves
fig_curves = go.Figure()

# Plot Initial Prior State
fig_curves.add_trace(go.Scatter(
    x=theta_grid, y=current_posterior.copy(), mode='lines',
    name='Prior State (k=0)',
    line=dict(dash='dash', width=2.5, color='gray')
))

# =====================================================================
# SEQUENTIAL BAYESIAN MONITORING LOOP
# =====================================================================
for k in range(1, n_sensor_readings + 1):
    # Simulate a noisy structural sensor reading from log-normal physics
    noise = np.random.normal(0, sigma)
    y_k = (theta_true * K_nominal) * np.exp(noise)

    # Calculate Log-Normal Likelihood curve across the structural theta grid
    # Expected value for any grid point is: grid_point * K_nominal
    expected_K = theta_grid * K_nominal
    likelihood = stats.lognorm.pdf(y_k, s=sigma, scale=expected_K)

    # Running Update: Posterior Proportional to Prior * Likelihood
    current_posterior = current_posterior * likelihood

    # Numerical Normalization via Trapezoidal rule
    integral = np.trapezoid(current_posterior, theta_grid)
    current_posterior /= integral

    # Calculate Point Estimators at step k
    theta_bayes_k = np.trapezoid(theta_grid * current_posterior, theta_grid)
    theta_map_k = theta_grid[np.argmax(current_posterior)]

    # Append point estimators to trackers
    bayes_estimates.append(theta_bayes_k)
    map_estimates.append(theta_map_k)

    # Capture structural health density profile at milestones
    if k in milestones:
        fig_curves.add_trace(go.Scatter(
            x=theta_grid, y=current_posterior.copy(), mode='lines',
            name=f"Step {k}: Post-Sensor (Observed K={y_k:.2f})",
            line=dict(width=2)
        ))

# =====================================================================
# PLOT 1: POSTERIOR DENSITY PROFILES
# =====================================================================
fig_curves.add_vline(
    x=theta_true, line_dash="dot", line_color="red", line_width=2.5,
    annotation_text=f"True State ({theta_true})",
    annotation_position="top left"
)

fig_curves.update_layout(
    title={
        'text': "Structural Health Monitoring: Bounded Bayesian Parameter Updating",
        'y': 0.95, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Remaining Structural Stiffness Efficiency Factor (θ)",
    yaxis_title="Probability Density (Confidence level of damage)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top", y=0.95, xanchor="left", x=0.02,
        bgcolor="rgba(255,255,255,0.7)"
    )
)

# =====================================================================
# PLOT 2: ESTIMATOR CONVERGENCE TIMELINE
# =====================================================================
steps = list(range(n_sensor_readings + 1))
fig_timeline = go.Figure()

# Add true state reference line
fig_timeline.add_hline(
    y=theta_true, line_dash="dash", line_color="red", line_width=2,
    annotation_text=f"True State (\u03b8_true = {theta_true})",
    annotation_position="bottom right"
)

# Trace for Posterior Mean
fig_timeline.add_trace(go.Scatter(
    x=steps, y=bayes_estimates, mode='lines+markers',
    name="Posterior Mean (\u03b8_Bayes)",
    line=dict(color="blue", width=2.5),
    marker=dict(size=6)
))

# Trace for MAP Estimate
fig_timeline.add_trace(go.Scatter(
    x=steps, y=map_estimates, mode='lines+markers',
    name="MAP Estimate (\u03b8_MAP)",
    line=dict(color="green", width=2, dash="dot"),
    marker=dict(size=6, symbol="square")
))

fig_timeline.update_layout(
    title={
        'text': "Convergence of Point Estimators (Mean & MAP) vs. True State",
        'y': 0.95, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Inspection Time Step (k)",
    yaxis_title="Estimated Stiffness Factor (θ)",
    xaxis=dict(tickmode="linear", tick0=0, dtick=1),
    yaxis=dict(range=[0.5, 1.05]),
    template="plotly_white",
    legend=dict(
        yanchor="top", y=0.95, xanchor="right", x=0.98,
        bgcolor="rgba(255,255,255,0.7)"
    )
)

# Display both interactive figures
fig_curves.show()
fig_timeline.show()

### Analysis of Convergence

*   **Prior Override Speed:** It takes exactly **3 to 4 sensor readings** for the system to overcome the initially optimistic, "healthy" $\text{Beta}(8, 1.5)$ prior. By step $k = 5$, the posterior distribution is fully centered over the actual damage state, and both the running Posterior Mean and MAP converge closely to $\theta_{\text{true}} = 0.68$.
*   **Safety Threshold Implications:** The systematic narrowing of the posterior density curves as $k \to 15$ signifies a rapid reduction in posterior variance (higher structural certainty). For structural safety thresholds, this behavior allows engineers to narrow the confidence bands around physical state estimates, significantly reducing false alarm rates while maintaining early-stage detection sensitivity.

## 1. Deriving the Marginal Density


$$p(x_i) = \sum_{k=1}^K p(x_i, C_i = k)$$



$$p(x_i) = \sum_{k=1}^K P(C_i = k) p(x_i \mid C_i = k)$$



$$p(x_i) = \sum_{k=1}^K \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)$$

### Why It Is Called a Gaussian Mixture Density

This density is called a **Gaussian mixture density** because it is a linear combination (a "mixture") of $K$ distinct Gaussian component densities. The mixing coefficients $\phi_k$ act as weighting factors. Because they satisfy the conditions $\phi_k \geq 0$ and $\sum_{k=1}^K \phi_k = 1$, the resulting function $p(x_i)$ remains a mathematically valid probability density function that integrates to $1$ over $\mathbb{R}^d$.

## 2. Deriving the Posterior Cluster Probability



$$P(C_i = k \mid X_i = x_i) = \frac{p(X_i = x_i \mid C_i = k) P(C_i = k)}{p(x_i)}$$



$$P(C_i = k \mid X_i = x_i) = \frac{p(X_i = x_i \mid C_i = k) P(C_i = k)}{\sum_{j=1}^K p(X_i = x_i \mid C_i = j) P(C_i = j)}$$


$$\gamma_{ik} = P(C_i = k \mid X_i = x_i) = \frac{\phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \phi_j \mathcal{N}(x_i \mid \mu_j, \Sigma_j)}$$

### Interpretation of $\gamma_{ik}$

The quantity $\gamma_{ik}$ is interpreted as the **posterior probability of cluster membership** because it updates our initial prior belief $\phi_k$ using the empirical evidence provided by the spatial location of $x_i$.

It scales the prior weight of cluster $k$ by the likelihood that a Gaussian centered at $\mu_k$ with covariance $\Sigma_k$ would generate a point at $x_i$, normalized across all $K$ possible clusters to ensure that $\sum_{k=1}^K \gamma_{ik} = 1$.

## 3. One-Hot Encoding of the Latent Cluster Variable

Let $Z_i = [Z_{i1}, Z_{i2}, \dots, Z_{iK}]^T$ be a one-hot encoded vector representing cluster membership, where:

$$Z_{ik} = \begin{cases} 1 & \text{if } C_i = k \\ 0 & \text{otherwise} \end{cases}$$

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = \sum_{z \in \{0,1\}} z \cdot P(Z_{ik} = z \mid X_i = x_i)$$

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = 1 \cdot P(Z_{ik} = 1 \mid X_i = x_i) + 0 \cdot P(Z_{ik} = 0 \mid X_i = x_i)$$

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = P(C_i = k \mid X_i = x_i) = \gamma_{ik}$$



$$\mathbb{E}[Z_i \mid X_i = x_i] = \begin{bmatrix} \mathbb{E}[Z_{i1} \mid X_i = x_i] \\ \mathbb{E}[Z_{i2} \mid X_i = x_i] \\ \vdots \\ \mathbb{E}[Z_{iK} \mid X_i = x_i] \end{bmatrix} = \begin{bmatrix} \gamma_{i1} \\ \gamma_{i2} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$

### Conclusion

This proves that the **soft cluster assignment vector** in a Gaussian Mixture Model is mathematically identical to the conditional expectation of the latent indicator vector, $\mathbb{E}[Z_i \mid X_i = x_i]$.

## 4. From Soft Assignment to Hard Clustering

In GMMs, we can transition from a probabilistic representation to a deterministic partition:

*   **Soft Clustering:** Represents assignments as continuous vectors of probabilities $\mathbb{E}[Z_i \mid X_i = x_i] = [\gamma_{i1}, \dots, \gamma_{iK}]^T$, where $\gamma_{ik} \in [0, 1]$ and $\sum_{k=1}^K \gamma_{ik} = 1$. This preserves classification uncertainty, showing if a point lies in an overlapping boundary region between clusters.
*   **Hard Clustering:** Places each point into a single, mutually exclusive category $\hat{C}_i$ by selecting the component with the highest posterior probability:

$$\hat{C}_i = \arg\max_{1 \leq k \leq K} \gamma_{ik}$$



This discards uncertainty by projecting the continuous probability vector onto a discrete one-hot coordinate vector (e.g., converting $[0.1, 0.7, 0.2]^T$ to $[0, 1, 0]^T$).

## 5. Conditional Expectation of the Observation Given the Cluster

Since the conditional distribution of $X_i$ given that it belongs to cluster $k$ is $\mathcal{N}(\mu_k, \Sigma_k)$, the conditional expectation of the observation is:

$$\mathbb{E}[X_i \mid C_i = k] = \int_{\mathbb{R}^d} x_i \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \, dx_i = \mu_k$$

### Interpreting and Comparing the Expectations

*   **Why $\mu_k$ is the center:** The mean vector $\mu_k$ acts as the spatial center of gravity of the $k$-th Gaussian component. It represents the prototypical coordinate profile of a data point generated by that specific cluster.
*   **Comparison:**
*   $\mathbb{E}[Z_i \mid X_i = x_i]$ operates in **probability space** ($\mathbb{R}^K$). It maps a known, observed data point $x_i$ to its fractional, posterior probabilities of belonging to each of the $K$ groups.
*   $\mathbb{E}[X_i \mid C_i = k]$ operates in **feature space** ($\mathbb{R}^d$). It maps a discrete cluster identity $k$ to its expected physical location in the multi-dimensional feature space.

## 6. The Complete-Data Likelihood

When both the observations $x_i$ and their latent cluster assignments $z_i$ are known, the complete-data joint likelihood is written as:

$$p(X=x, Z=z \mid \Theta) = \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}}$$


$$\ell_c = \ln p(X=x, Z=z \mid \Theta) = \sum_{i=1}^n \sum_{k=1}^K \ln \left( \left[ \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}} \right)$$

$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \ln \phi_k + \ln \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

### Why This Is Easy to Maximize

If the latent indicators $z_{ik}$ were known, this optimization problem would decouple into $K$ independent sub-problems.

Instead of dealing with sums inside logarithms (as in the marginal log-likelihood), the log-likelihood for each component $k$ simplifies to standard maximum likelihood estimation (MLE) for a single Gaussian, using only the subset of data points where $z_{ik} = 1$:

$$\mu_k = \frac{\sum_{i=1}^n z_{ik} x_i}{\sum_{i=1}^n z_{ik}}, \quad \phi_k = \frac{\sum_{i=1}^n z_{ik}}{n}$$

## 7. The EM Interpretation

In practical applications, the exact values of the latent variables $Z_i$ are hidden. The Expectation-Maximization (EM) algorithm resolves this by calculating the conditional expectation of the complete-data log-likelihood with respect to the posterior distribution of the latent variables, given the current parameter estimates $\Theta^{(t)}$:

$$Q(\Theta \mid \Theta^{(t)}) = \mathbb{E}_{Z \mid X, \Theta^{(t)}} [\ell_c]$$

Because the complete-data log-likelihood $\ell_c$ is a linear function of the latent variables $z_{ik}$, we pass the expectation operator inside the summation:

$$Q(\Theta \mid \Theta^{(t)}) = \sum_{i=1}^n \sum_{k=1}^K \mathbb{E}[Z_{ik} \mid X_i = x_i, \Theta^{(t)}] \left[ \ln \phi_k + \ln \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

Substituting $\mathbb{E}[Z_{ik} \mid X_i = x_i, \Theta^{(t)}] = \gamma_{ik}$ yields the surrogate objective function minimized in the M-step:

$$Q(\Theta \mid \Theta^{(t)}) = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \left[ \ln \phi_k + \ln \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

### E-Step as a Conditional Update

The Expectation step (E-step) is a **conditional update** because it computes the conditional distribution of the unobserved latent cluster variables based on the current parameter estimates and the observed data. It updates the assignment weights of each point to reflect the current state of the model.

## 8. Parameter Updates

To derive the parameter updates for the M-step, we maximize $Q(\Theta \mid \Theta^{(t)})$ with respect to each set of parameters.

### 1. Updating the Mixing Weights $\phi_k$



$$\mathcal{L}(\phi, \lambda) = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \ln \phi_k + \lambda \left( 1 - \sum_{k=1}^K \phi_k \right)$$



$$\frac{\partial \mathcal{L}}{\partial \phi_k} = \sum_{i=1}^n \frac{\gamma_{ik}}{\phi_k} - \lambda = 0 \implies \phi_k = \frac{\sum_{i=1}^n \gamma_{ik}}{\lambda}$$



$$\sum_{k=1}^K \phi_k = \sum_{k=1}^K \frac{\sum_{i=1}^n \gamma_{ik}}{\lambda} \implies 1 = \frac{\sum_{i=1}^n \sum_{k=1}^K \gamma_{ik}}{\lambda}$$

$\sum_{k=1}^K \gamma_{ik} = 1$,

$$\phi_k^{\text{new}} = \frac{N_k}{n}, \quad \text{where } N_k = \sum_{i=1}^n \gamma_{ik}$$

### 2. Updating the Means $\mu_k$



$$Q(\mu_k) = \sum_{i=1}^n \gamma_{ik} \left[ -\frac{1}{2} (x_i - \mu_k)^T \Sigma_k^{-1} (x_i - \mu_k) \right] + \text{constant}$$


$$\frac{\partial Q}{\partial \mu_k} = \sum_{i=1}^n \gamma_{ik} \Sigma_k^{-1} (x_i - \mu_k) = 0$$



$$\sum_{i=1}^n \gamma_{ik} (x_i - \mu_k) = 0 \implies \sum_{i=1}^n \gamma_{ik} x_i = \mu_k \sum_{i=1}^n \gamma_{ik}$$

$$\mu_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} x_i$$

### 3. Updating the Covariances $\Sigma_k$

$$Q(\Sigma_k) = \sum_{i=1}^n \gamma_{ik} \left[ -\frac{1}{2} \ln \vert{}\Sigma_k\vert{} - \frac{1}{2} (x_i - \mu_k)^T \Sigma_k^{-1} (x_i - \mu_k) \right]$$


$$\frac{\partial Q}{\partial \Sigma_k^{-1}} = \sum_{i=1}^n \gamma_{ik} \left[ \frac{1}{2} \Sigma_k - \frac{1}{2} (x_i - \mu_k)(x_i - \mu_k)^T \right] = 0$$

$$\Sigma_k \sum_{i=1}^n \gamma_{ik} = \sum_{i=1}^n \gamma_{ik} (x_i - \mu_k)(x_i - \mu_k)^T$$

$$\Sigma_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} (x_i - \mu^{\text{new}}_k)(x_i - \mu^{\text{new}}_k)^T$$

### How the Responsibility Acts as a Weight

In these update rules, the responsibility $\gamma_{ik}$ acts as a **fractional membership weight**. Instead of a data point contributing fully to only one cluster (as in $K$-means, where weights are binary $\{0, 1\}$), each observation $x_i$ contributes fractionally to all $K$ clusters.

A point with a high responsibility $\gamma_{ik} \approx 1$ strongly shapes the mean and covariance of cluster $k$, while a point on a cluster boundary with $\gamma_{ik} \approx 0.5$ contributes equally to the updates of both neighboring clusters.

## 9. GMM Clustering as an Iterative Updating Process

Gaussian Mixture Model clustering can be viewed as an iterative process of conditional updating.

At the start of each iteration, the mixture weight $\phi_k$ represents our prior probability of selecting cluster $k$. For any given data point $x_i$, the Gaussian density function $\mathcal{N}(x_i \mid \mu_k, \Sigma_k)$ acts as the likelihood, measuring how compatible the point's coordinates are with the shape and center of cluster $k$.

By applying Bayes' rule, the E-step combines this prior and likelihood to compute the responsibility $\gamma_{ik}$, which is the posterior probability of membership in cluster $k$ after observing the point $x_i$. These individual probabilities form the components of the soft assignment vector $\mathbb{E}[Z_i \mid X_i = x_i]$.

Finally, the M-step uses these updated posterior probabilities as fractional weights to recalculate the cluster parameters (weights, means, and covariances), aligning the clusters with the data.

Thus, GMM clustering is a process of probabilistic clustering that iteratively refines its parameters based on the conditional expectations of latent variables.

## 10. Computational Simulation and Out-of-Sample Validation

In [8]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

class GMMFinancialSegmenter:
    def __init__(self, n_components=3, random_state=42):
        """
        Initializes the GMM financial segmenter.
        """
        self.n_components = n_components
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.model = GaussianMixture(
            n_components=self.n_components,
            covariance_type="full",
            random_state=self.random_state
        )
        self.feature_cols = None

    def prepare_data(self, df, feature_cols, test_size=0.2):
        """
        Extracts, cleans, standardizes, and splits the data.
        """
        self.feature_cols = feature_cols
        # Drop missing records across our target features
        df_cleaned = df[feature_cols].dropna()
        X = df_cleaned.values

        # Scale features
        X_scaled = self.scaler.fit_transform(X)

        # Split into Train (80%) and Test (20%) sets
        X_train, X_test = train_test_split(
            X_scaled,
            test_size=test_size,
            random_state=self.random_state
        )
        return X_train, X_test

    def fit(self, X_train):
        """
        Trains the Gaussian Mixture Model using EM.
        """
        self.model.fit(X_train)
        print("=========================================")
        print("GMM Training Summary:")
        print(f"Model Converged: {self.model.converged_}")
        print(f"Iterations Required: {self.model.n_iter_}")
        print("=========================================")

    def evaluate(self, X_test):
        """
        Computes the average log-likelihood on unseen test data.
        """
        avg_log_likelihood = self.model.score(X_test)
        print(f"Average Log-Likelihood on Test Set: {avg_log_likelihood:.4f}")
        return avg_log_likelihood

    def plot_density_heatmap(self, X_train):
        """
        Generates a 2D density heatmap of the raw empirical training data.
        """
        # Revert scaling for realistic coordinate axes
        X_orig = self.scaler.inverse_transform(X_train)

        fig = px.density_heatmap(
            x=X_orig[:, 0],
            y=X_orig[:, 1],
            marginal_x="histogram",
            marginal_y="histogram",
            labels={"x": self.feature_cols[0], "y": self.feature_cols[1]},
            title="Empirical Training Data 2D Density Heatmap (with Marginal Distributions)",
            color_continuous_scale=px.colors.sequential.Viridis  # Uses the list of colors to bypass Plotly Express bug
        )
        fig.update_layout(template="plotly_white")
        fig.show()

    def _get_contour_grid(self, X_data):
        """
        Helper method to generate coordinate grids and predict posterior responsibilities.
        """
        x_min, x_max = X_data[:, 0].min() - 0.5, X_data[:, 0].max() + 0.5
        y_min, y_max = X_data[:, 1].min() - 0.5, X_data[:, 1].max() + 0.5

        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
        grid_points = np.c_[xx.ravel(), yy.ravel()]

        # Calculate responsibilities for all grid points
        responsibilities = self.model.predict_proba(grid_points)
        # Find the maximum posterior probability at each coordinate
        max_prob = responsibilities.max(axis=1).reshape(xx.shape)

        # Convert grid back to original coordinate system for plotting
        grid_orig = self.scaler.inverse_transform(grid_points)
        xx_orig = grid_orig[:, 0].reshape(xx.shape)
        yy_orig = grid_orig[:, 1].reshape(yy.shape)

        return xx_orig, yy_orig, max_prob

    def plot_assignments(self, X_data, title_prefix="Training"):
        """
        Plots data points over the soft assignment confidence contours.
        """
        xx_orig, yy_orig, max_prob = self._get_contour_grid(X_data)
        hard_labels = self.model.predict(X_data)
        X_orig = self.scaler.inverse_transform(X_data)

        fig = go.Figure()

        # Add background contours representing assignment confidence levels
        fig.add_trace(go.Contour(
            x=xx_orig[0, :],
            y=yy_orig[:, 0],
            z=max_prob,
            colorscale="Cividis",
            contours_coloring="heatmap",
            name="Responsibility Confidence",
            hoverinfo="skip",
            opacity=0.85,
            colorbar=dict(title="Max Posterior Prob")
        ))

        # Add scatter traces for each assigned cluster
        for k in range(self.n_components):
            mask = (hard_labels == k)
            fig.add_trace(go.Scatter(
                x=X_orig[mask, 0],
                y=X_orig[mask, 1],
                mode="markers",
                name=f"Cluster {k+1}",
                marker=dict(
                    size=5,
                    line=dict(width=0.5, color="white")
                )
            ))

        fig.update_layout(
            title=f"GMM Soft-Assignment Confidence Boundaries ({title_prefix} Set)",
            xaxis_title=self.feature_cols[0],
            yaxis_title=self.feature_cols[1],
            template="plotly_white",
            legend=dict(yanchor="top", y=0.98, xanchor="left", x=0.02, bgcolor="rgba(255,255,255,0.6)")
        )
        fig.show()

# =====================================================================
# EXECUTION PIPELINE
# =====================================================================
if __name__ == "__main__":
    # Check for local copy of Kaggle CC GENERAL dataset
    csv_path = "CC GENERAL.csv"

    if os.path.exists(csv_path):
        print(f"Loading local dataset from: {csv_path}")
        df = pd.read_csv(csv_path)
    else:
        print("Local Kaggle dataset not found. Generating representative synthetic data...")
        # Simulating PURCHASES and CREDIT_LIMIT columns
        np.random.seed(42)
        n_samples = 1000

        # Synthesize three customer segments
        purchases_segment_1 = np.random.exponential(300, int(n_samples * 0.5))
        credit_segment_1 = np.random.normal(1500, 500, int(n_samples * 0.5))

        purchases_segment_2 = np.random.normal(2500, 600, int(n_samples * 0.35))
        credit_segment_2 = np.random.normal(6500, 1200, int(n_samples * 0.35))

        purchases_segment_3 = np.random.normal(7000, 1500, int(n_samples * 0.15))
        credit_segment_3 = np.random.normal(12000, 2000, int(n_samples * 0.15))

        df = pd.DataFrame({
            "PURCHASES": np.concatenate([purchases_segment_1, purchases_segment_2, purchases_segment_3]),
            "CREDIT_LIMIT": np.concatenate([credit_segment_1, credit_segment_2, credit_segment_3])
        })

    # Configure Segmenter
    target_features = ["PURCHASES", "CREDIT_LIMIT"]
    segmenter = GMMFinancialSegmenter(n_components=3, random_state=42)

    # Process and Split
    X_train, X_test = segmenter.prepare_data(df, target_features)

    # Fit Model
    segmenter.fit(X_train)

    # Evaluate Out-of-Sample Performance
    segmenter.evaluate(X_test)

    # Generate Plots
    segmenter.plot_density_heatmap(X_train)
    segmenter.plot_assignments(X_train, title_prefix="Training")
    segmenter.plot_assignments(X_test, title_prefix="Validation / Test")

Local Kaggle dataset not found. Generating representative synthetic data...
GMM Training Summary:
Model Converged: True
Iterations Required: 3
Average Log-Likelihood on Test Set: -0.7477


## 11. Plot Evaluation and Mathematical Synthesis

### Plot Analysis

*   **Empirical 2D Density Heatmap:** This plot reveals a clear multimodal structure. Most of the data is concentrated in the lower-left corner (representing low-spending customers with lower credit limits). A second, broader mode stretches toward moderate purchases and credit limits, while a sparse, high-value group occupies the upper-right region.
*   **Assignment Plots (Training and Test):** The scatter plots show how the model divides the customer base. The background contour lines display the maximum posterior probability, $\max_k \gamma_{ik}$.

### Connection to $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$

The continuous background contour map provides a visual representation of the soft assignment expectation vector:

$$\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}] = [\gamma_{i1}, \gamma_{i2}, \dots, \gamma_{iK}]^T$$

*   **High-Confidence Regions:** In the centers of each cluster, the contour values approach $1.0$ (shown in dark, solid colors). This indicates that the conditional expectation vector is close to a one-hot coordinate vector (e.g., $[0.99, 0.005, 0.005]^T$), indicating a high-confidence assignment.
*   **Boundary Ambiguity:** In the transition zones between clusters, the contour levels drop significantly (shown as lighter bands). Along these boundaries, the components of the conditional expectation vector are nearly equal (e.g., $[0.48, 0.48, 0.04]^T$, where the maximum probability drops to around $0.5$).

This contour map visually demonstrates how GMM clustering preserves the uncertainty in the conditional expectation vector, mapping the continuous transition from one cluster to another across the feature space.